In [ ]:

import importlib
import subprocess
import sys

def _ensure(package, pip_name=None):
    try:
        importlib.import_module(package)
    except ImportError:
        pip_name = pip_name or package
        print(f"[setup] Installing missing package: {pip_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

_ensure("sentencepiece")
_ensure("google.protobuf", pip_name="protobuf")

import os
import json
import random
import time
import gc
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    get_linear_schedule_with_warmup,
    BitsAndBytesConfig,
)
from sklearn.metrics import (
    classification_report,
    f1_score,
    precision_recall_fscore_support,
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training,
)

# ============================================================================
# CONFIGURATION
# ============================================================================

MISTRAL_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

TRAIN_PATH = "build_jsonl/build_train.jsonl"
DEV_PATH   = "build_jsonl/build_dev.jsonl"
TEST_PATH  = "build_jsonl/build_test.jsonl"

OUT_DIR = "mistral_legal_classification"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
MAX_SEQ_LENGTH = 512
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4

USE_QLORA = True
QLORA_BITS = 4
QLORA_COMPUTE_DTYPE = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
)

LORA_R = 64
LORA_ALPHA = 128
LORA_DROPOUT = 0.1
MISTRAL_LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

NUM_EPOCHS = 20
LR = 2e-4
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
WARMUP_RATIO = 0.05

FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.25
LABEL_SMOOTHING = 0.02
PROTO_WEIGHT = 0.1

USE_WEIGHTED_SAMPLER = True
MINORITY_BOOST = 3.0

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {label: idx for idx, label in enumerate(LABELS)}
id2label   = {idx: label for label, idx in label2id.items()}
NUM_LABELS = len(LABELS)

MINORITY_CLASSES = ["RLC", "ISSUE", "STA", "RATIO", "PRE_RELIED", "PRE_NOT_RELIED", "RPC"]
minority_ids = [label2id[l] for l in MINORITY_CLASSES if l in label2id]

# ============================================================================
#  FIX 1 — Auto-detect primary device; never hardcode "cuda:1"
# ============================================================================
def get_primary_device() -> torch.device:
    """
    Returns cuda:0 if any GPU is available, else cpu.
    Never uses cuda:1 unless you explicitly have 2+ GPUs.
    """
    if torch.cuda.is_available():
        return torch.device("cuda:0")
    return torch.device("cpu")

PRIMARY_DEVICE = get_primary_device()
print(f"[Config] Primary device: {PRIMARY_DEVICE}")
print(f"[Config] GPU count: {torch.cuda.device_count()}")

# ============================================================================
# UTILITIES
# ============================================================================

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data


def extract_data(docs, max_sents=64):
    all_sents, all_labels, doc_ids = [], [], []
    for doc in docs:
        doc_id = doc.get("id", "")
        sents, labs = [], []

        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val       = item.get("value", {})
                    text      = val.get("text", "").strip()
                    labs_list = val.get("labels", ["NONE"])
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(labs_list[0], label2id["NONE"]))

        if len(sents) > max_sents:
            sents = sents[:max_sents]
            labs  = labs[:max_sents]

        if sents and labs and len(sents) == len(labs):
            all_sents.append(sents)
            all_labels.append(labs)
            doc_ids.append(doc_id)

    return all_sents, all_labels, doc_ids


def compute_detailed_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0, labels=range(NUM_LABELS)
    )
    macro_f1    = f1_score(y_true, y_pred, average="macro",    zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    minority_mask = np.isin(y_true, minority_ids)
    minority_f1   = 0.0
    if minority_mask.sum() > 0:
        minority_f1 = f1_score(
            y_true[minority_mask], y_pred[minority_mask],
            average="macro", zero_division=0,
        )

    return {
        "macro_f1":          float(macro_f1),
        "weighted_f1":       float(weighted_f1),
        "minority_macro_f1": float(minority_f1),
        "per_class_f1":      {id2label[i]: float(f1[i])    for i in range(NUM_LABELS)},
        "per_class_support": {id2label[i]: int(support[i]) for i in range(NUM_LABELS)},
    }

# ============================================================================
#  FIX 2 — Prototype Manager: resolve device from model, not from config
# ============================================================================

class ClassPrototypeManager:
    def __init__(self):
        self.prototypes = None
        self.fitted     = False

    def fit(self, embeddings, labels):
        embeddings = np.asarray(embeddings)
        labels     = np.asarray(labels)
        D          = embeddings.shape[1]
        protos     = np.zeros((NUM_LABELS, D), dtype=np.float32)
        for k in range(NUM_LABELS):
            mask = labels == k
            protos[k] = (
                embeddings[mask].mean(axis=0)
                if mask.sum() > 0
                else np.random.randn(D).astype(np.float32) * 1e-3
            )
        self.prototypes = protos
        self.fitted     = True
        print(f"[Prototypes] Fitted with shape {protos.shape}")

    def get_all_tensor(self, device: torch.device) -> torch.Tensor:
        """
         FIX 2: caller must pass the actual resolved device,
        not a hardcoded string like 'cuda:1'.
        """
        if not self.fitted:
            raise RuntimeError("Prototypes not fitted — call fit() first")
        return torch.tensor(self.prototypes, dtype=torch.float32).to(device)

# ============================================================================
# DATASET
# ============================================================================

class MistralLegalDataset(Dataset):
    def __init__(self, docs_sents, docs_labels, tokenizer,
                 max_length=MAX_SEQ_LENGTH):
        self.docs_sents  = docs_sents
        self.docs_labels = docs_labels
        self.tokenizer   = tokenizer
        self.max_length  = max_length

    def __len__(self):
        return len(self.docs_sents)

    def __getitem__(self, idx):
        sentences = self.docs_sents[idx]
        labels    = self.docs_labels[idx]
        doc_text  = " ".join(f"[S{i}] {s}" for i, s in enumerate(sentences))
        return {
            "text":      doc_text,
            "sentences": sentences,
            "labels":    labels,
            "num_sents": len(sentences),
        }


def collate_mistral_batch(batch, tokenizer, max_length=MAX_SEQ_LENGTH):
    texts          = [b["text"]      for b in batch]
    all_sentences  = [b["sentences"] for b in batch]
    all_labels     = [b["labels"]    for b in batch]
    num_sents_list = [b["num_sents"] for b in batch]

    encoding = tokenizer(
        texts, padding=True, truncation=True,
        max_length=max_length, return_tensors="pt",
    )

    max_sents          = max(num_sents_list)
    all_sent_encodings = []
    for sents in all_sentences:
        sent_enc = tokenizer(
            sents, padding="max_length", truncation=True,
            max_length=128, return_tensors="pt",
        )
        if len(sents) < max_sents:
            pad = max_sents - len(sents)
            T   = sent_enc["input_ids"].size(1)
            sent_enc["input_ids"] = torch.cat(
                [sent_enc["input_ids"], torch.zeros(pad, T, dtype=torch.long)])
            sent_enc["attention_mask"] = torch.cat(
                [sent_enc["attention_mask"], torch.zeros(pad, T, dtype=torch.long)])
        all_sent_encodings.append(sent_enc)

    sent_input_ids      = torch.stack([e["input_ids"]      for e in all_sent_encodings])
    sent_attention_mask = torch.stack([e["attention_mask"] for e in all_sent_encodings])

    labels_padded = torch.full((len(batch), max_sents), -100, dtype=torch.long)
    for i, labs in enumerate(all_labels):
        labels_padded[i, : len(labs)] = torch.tensor(labs, dtype=torch.long)

    return {
        "doc_input_ids":      encoding["input_ids"],
        "doc_attention_mask": encoding["attention_mask"],
        "sent_input_ids":     sent_input_ids,
        "sent_attention_mask":sent_attention_mask,
        "labels":             labels_padded,
        "num_sents":          torch.tensor(num_sents_list, dtype=torch.long),
    }

# ============================================================================
#  FIX 3-6 — MistralLegalClassifier: full device-safe implementation
# ============================================================================

class MistralLegalClassifier(nn.Module):
    """
    Mistral-7B backbone (QLoRA) + sentence-level classification head.

    Device strategy
    ---------------
    • self.mistral uses device_map="auto" — accelerate places layers
      across available GPUs automatically.  We NEVER call .to() on it.
    • self.context_aggregator and self.classifier live on PRIMARY_DEVICE.
    • All intermediate tensors are explicitly moved to the device of the
      tensor they are combined with, using .to(tensor.device).
    """

    def __init__(
        self,
        model_name   = MISTRAL_MODEL_NAME,
        num_labels   = NUM_LABELS,
        use_qlora    = USE_QLORA,
        dropout      = 0.3,
        head_device  = PRIMARY_DEVICE,   #  FIX 3: explicit head device
    ):
        super().__init__()
        self.head_device = head_device

        # ── Mistral backbone ─────────────────────────────────────────────────
        if use_qlora:
            bnb_config = BitsAndBytesConfig(
                load_in_4bit              = True,
                bnb_4bit_use_double_quant = True,
                bnb_4bit_quant_type       = "nf4",
                bnb_4bit_compute_dtype    = QLORA_COMPUTE_DTYPE,
            )
            self.mistral = AutoModelForCausalLM.from_pretrained(
                model_name,
                quantization_config = bnb_config,
                device_map          = "auto",   # accelerate handles placement
                trust_remote_code   = True,
            )
            self.mistral = prepare_model_for_kbit_training(self.mistral)
        else:
            self.mistral = AutoModelForCausalLM.from_pretrained(
                model_name,
                device_map     = "auto",
                torch_dtype    = QLORA_COMPUTE_DTYPE,
                trust_remote_code = True,
            )

        lora_config  = LoraConfig(
            r              = LORA_R,
            lora_alpha     = LORA_ALPHA,
            lora_dropout   = LORA_DROPOUT,
            bias           = "none",
            task_type      = TaskType.CAUSAL_LM,
            target_modules = MISTRAL_LORA_TARGET_MODULES,
        )
        self.mistral = get_peft_model(self.mistral, lora_config)
        self.mistral.print_trainable_parameters()

        self.hidden_size = self.mistral.config.hidden_size  # 4096

        # ── Classification heads — placed on head_device ─────────────────────
        #  FIX 5: explicit placement; not left to accident
        self.context_aggregator = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model         = self.hidden_size,
                nhead           = 8,
                dim_feedforward = self.hidden_size * 2,
                dropout         = dropout,
                batch_first     = True,
                dtype           = torch.float32,
            ),
            num_layers = 2,
        ).to(self.head_device)

        self.proto_proj = nn.Linear(
            self.hidden_size, self.hidden_size
        ).to(self.head_device)

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.hidden_size, self.hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(self.hidden_size // 2, num_labels),
        ).to(self.head_device)

    # ── helpers ──────────────────────────────────────────────────────────────

    def _first_backbone_device(self) -> torch.device:
        """Returns the device holding the first parameter of the backbone."""
        try:
            return next(self.mistral.parameters()).device
        except StopIteration:
            return self.head_device

    def get_sentence_embeddings(
        self,
        input_ids:      torch.Tensor,   # [B, S, L]
        attention_mask: torch.Tensor,   # [B, S, L]
    ) -> torch.Tensor:                  # [B, S, H]  on head_device
        """
         FIX 4 & 6: inputs routed to backbone's first device;
        output moved to head_device before returning.
        """
        B, S, L       = input_ids.shape
        backbone_dev  = self._first_backbone_device()

        ids_flat  = input_ids.view(B * S, L).to(backbone_dev)
        mask_flat = attention_mask.view(B * S, L).to(backbone_dev)

        base_model = (
            self.mistral.model
            if hasattr(self.mistral, "model")
            else self.mistral
        )

        outputs     = base_model(
            input_ids      = ids_flat,
            attention_mask = mask_flat,
            output_hidden_states = True,
        )
        last_hidden = outputs.hidden_states[-1]   # [B*S, L, H]

        # Align mask to wherever the final hidden state landed
        mask_aligned  = mask_flat.to(last_hidden.device)
        mask_exp      = mask_aligned.unsqueeze(-1).expand(last_hidden.size()).float()
        sum_emb       = torch.sum(last_hidden * mask_exp, dim=1)
        sum_mask      = torch.clamp(mask_exp.sum(dim=1), min=1e-9)
        sent_emb_flat = (sum_emb / sum_mask).float()          # [B*S, H]

        #  FIX 6: move to head_device so classifier layers can use it
        return sent_emb_flat.to(self.head_device).view(B, S, self.hidden_size)

    # ── forward ──────────────────────────────────────────────────────────────

    def forward(
        self,
        doc_input_ids:      torch.Tensor,
        doc_attention_mask: torch.Tensor,
        sent_input_ids:     torch.Tensor,
        sent_attention_mask:torch.Tensor,
        num_sents:          torch.Tensor,
        prototypes:         torch.Tensor = None,
    ):
        B, S, L = sent_input_ids.shape

        # [B, S, H] — already on head_device after get_sentence_embeddings
        sent_emb = self.get_sentence_embeddings(sent_input_ids, sent_attention_mask)

        # Prototype attention residual
        if prototypes is not None:
            #  FIX 7: ensure prototypes are on head_device
            prototypes = prototypes.to(self.head_device)

            proto_proj      = self.proto_proj(sent_emb)               # [B, S, H]
            proto_proj_norm = F.normalize(proto_proj,  dim=-1)
            proto_norm      = F.normalize(prototypes,  dim=-1)        # [C, H]
            proto_sim       = torch.matmul(proto_proj_norm, proto_norm.t())  # [B, S, C]
            proto_context   = torch.matmul(proto_sim, prototypes)     # [B, S, H]
            sent_emb        = sent_emb + 0.1 * proto_context

        # Padding mask for TransformerEncoder
        num_sents_dev = num_sents.to(self.head_device)
        sent_mask = (
            torch.arange(S, device=self.head_device).unsqueeze(0)
            < num_sents_dev.unsqueeze(1)
        )  # [B, S]  True = valid

        contextualized = self.context_aggregator(
            sent_emb, src_key_padding_mask=~sent_mask
        )  # [B, S, H]

        logits = self.classifier(contextualized)  # [B, S, num_labels]

        return logits, sent_emb

# ============================================================================
# LOSS FUNCTIONS
# ============================================================================

def focal_loss(logits, labels,
               gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA,
               label_smoothing=LABEL_SMOOTHING):
    ce    = F.cross_entropy(logits, labels, reduction="none",
                            label_smoothing=label_smoothing)
    pt    = torch.exp(-ce)
    focal = alpha * (1 - pt) ** gamma * ce
    return focal.mean()


def compute_loss(logits, labels, sent_emb, prototypes,
                 proto_weight=PROTO_WEIGHT):
    logits_flat = logits.view(-1, NUM_LABELS)
    labels_flat = labels.view(-1)
    mask        = labels_flat != -100

    if mask.sum() == 0:
        return torch.tensor(0.0, device=logits.device, requires_grad=True)

    logits_masked = logits_flat[mask]
    labels_masked = labels_flat[mask]

    cls_loss = focal_loss(logits_masked, labels_masked)

    if prototypes is not None and proto_weight > 0:
        sent_emb_flat = sent_emb.view(-1, sent_emb.size(-1))[mask]
        sent_norm     = F.normalize(sent_emb_flat, dim=-1)
        proto_norm    = F.normalize(prototypes.to(sent_emb.device), dim=-1)
        proto_sim     = torch.matmul(sent_norm, proto_norm.t())
        proto_loss    = F.cross_entropy(proto_sim * 5.0, labels_masked)
        return cls_loss + proto_weight * proto_loss

    return cls_loss

# ============================================================================
#  FIX 3 & 4 — Trainer: no hardcoded device strings
# ============================================================================

class MistralTrainer:
    """
    Training loop that is fully agnostic to the number of GPUs.

    Key design:
      • prototype_tensor is created on PRIMARY_DEVICE (= cuda:0 or cpu)
      • _move_batch sends *input* tensors to PRIMARY_DEVICE;
        the backbone's device_map routes them further automatically
      • model.head_device == PRIMARY_DEVICE, so logits / sent_emb are
        always on the same device as labels and prototypes
    """

    def __init__(self, model: MistralLegalClassifier,
                 tokenizer,
                 prototype_manager: ClassPrototypeManager):
        #  FIX 3: NEVER call model.to(device) — device_map="auto" is in charge
        self.model             = model
        self.tokenizer         = tokenizer
        self.prototype_manager = prototype_manager
        self.primary_device    = model.head_device   # cuda:0 or cpu

    # ── helpers ──────────────────────────────────────────────────────────────

    def _get_prototypes(self) -> torch.Tensor:
        """Prototype tensor lives on head_device (= cuda:0 or cpu)."""
        return self.prototype_manager.get_all_tensor(device=self.primary_device)

    def _move_batch(self, batch: dict) -> dict:
        """
         FIX 4: move input tensors to PRIMARY_DEVICE only.
        The backbone's device_map will move activations to subsequent GPUs.
        """
        return {
            k: v.to(self.primary_device) if isinstance(v, torch.Tensor) else v
            for k, v in batch.items()
        }

    def _build_train_loader(self, dataset: MistralLegalDataset) -> DataLoader:
        cf = lambda b: collate_mistral_batch(b, self.tokenizer)

        if not USE_WEIGHTED_SAMPLER:
            return DataLoader(dataset, batch_size=BATCH_SIZE,
                              shuffle=True, collate_fn=cf)

        major_labels = [
            Counter(labs).most_common(1)[0][0] if labs else 0
            for labs in dataset.docs_labels
        ]
        counts   = np.bincount(major_labels, minlength=NUM_LABELS)
        inv_freq = 1.0 / (counts + 1e-6)
        weights  = inv_freq[np.array(major_labels)].copy()
        for i, ml in enumerate(major_labels):
            if ml in minority_ids:
                weights[i] *= MINORITY_BOOST

        sampler = WeightedRandomSampler(
            torch.tensor(weights, dtype=torch.double),
            num_samples = len(weights),
            replacement = True,
        )
        return DataLoader(dataset, batch_size=BATCH_SIZE,
                          sampler=sampler, collate_fn=cf)

    # ── train ────────────────────────────────────────────────────────────────

    def train(self, train_dataset, dev_dataset,
              num_epochs=NUM_EPOCHS, lr=LR):
        print("Starting Mistral-7B training...")

        optimizer    = torch.optim.AdamW(
            self.model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
        train_loader = self._build_train_loader(train_dataset)

        total_steps  = len(train_loader) * num_epochs
        warmup_steps = max(1, int(WARMUP_RATIO * total_steps))
        scheduler    = get_linear_schedule_with_warmup(
            optimizer, warmup_steps, total_steps)

        # Prototypes on head_device
        prototypes_tensor = self._get_prototypes()

        best_macro_f1 = -1.0
        best_ckpt     = None
        history       = []

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            epoch_start  = time.time()
            running_loss = 0.0
            n_samples    = 0

            optimizer.zero_grad()   #  FIX 9: zero before first step

            for step, batch in enumerate(train_loader):
                batch = self._move_batch(batch)

                logits, sent_emb = self.model(
                    batch["doc_input_ids"],
                    batch["doc_attention_mask"],
                    batch["sent_input_ids"],
                    batch["sent_attention_mask"],
                    batch["num_sents"],
                    prototypes_tensor,
                )

                loss = compute_loss(
                    logits, batch["labels"], sent_emb, prototypes_tensor
                )
                loss = loss / GRADIENT_ACCUMULATION_STEPS
                loss.backward()

                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(
                        self.model.parameters(), GRAD_CLIP)
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()

                valid_n       = (batch["labels"].view(-1) != -100).sum().item()
                running_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS * valid_n
                n_samples    += valid_n

                if (step + 1) % 10 == 0:
                    avg = running_loss / max(1, n_samples)
                    print(f"  Ep {epoch} Step {step+1}/{len(train_loader)} | "
                          f"Loss: {avg:.4f}")

            # Final gradient flush for incomplete accumulation window
            if (len(train_loader)) % GRADIENT_ACCUMULATION_STEPS != 0:
                torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(), GRAD_CLIP)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            avg_loss    = running_loss / max(1, n_samples)
            val_results = self.evaluate(dev_dataset)
            epoch_time  = time.time() - epoch_start

            history.append({
                "epoch":            epoch,
                "train_loss":       avg_loss,
                "val_macro_f1":     val_results["metrics"]["macro_f1"],
                "val_weighted_f1":  val_results["metrics"]["weighted_f1"],
                "val_minority_f1":  val_results["metrics"]["minority_macro_f1"],
                "epoch_time_s":     epoch_time,
            })

            print(f"\nEpoch {epoch}/{num_epochs} | "
                  f"Loss: {avg_loss:.4f} | "
                  f"Val MacroF1: {val_results['metrics']['macro_f1']:.4f} | "
                  f"MinorityF1: {val_results['metrics']['minority_macro_f1']:.4f} | "
                  f"Time: {epoch_time:.1f}s\n")

            if val_results["metrics"]["macro_f1"] > best_macro_f1:
                best_macro_f1 = val_results["metrics"]["macro_f1"]
                ckpt_path = os.path.join(
                    OUT_DIR,
                    f"best_epoch{epoch}_f1{best_macro_f1:.4f}.pt",
                )
                torch.save({
                    "model_state_dict": self.model.state_dict(),
                    "epoch":    epoch,
                    "macro_f1": best_macro_f1,
                }, ckpt_path)
                if best_ckpt and os.path.exists(best_ckpt):
                    os.remove(best_ckpt)
                best_ckpt = ckpt_path
                print(f"Saved best model: {ckpt_path}\n")

        pd.DataFrame(history).to_csv(
            os.path.join(OUT_DIR, "training_history.csv"), index=False)
        return best_ckpt

    # ── evaluate ─────────────────────────────────────────────────────────────

    def evaluate(self, dataset: MistralLegalDataset) -> dict:
        self.model.eval()
        loader = DataLoader(
            dataset, batch_size=BATCH_SIZE, shuffle=False,
            collate_fn=lambda b: collate_mistral_batch(b, self.tokenizer),
        )

        #  FIX 10: same device resolution as train()
        prototypes_tensor = self._get_prototypes()
        all_preds, all_trues = [], []

        with torch.no_grad():
            for batch in loader:
                batch = self._move_batch(batch)

                logits, _ = self.model(
                    batch["doc_input_ids"],
                    batch["doc_attention_mask"],
                    batch["sent_input_ids"],
                    batch["sent_attention_mask"],
                    batch["num_sents"],
                    prototypes_tensor,
                )

                logits_flat = logits.view(-1, NUM_LABELS)
                labels_flat = batch["labels"].view(-1)
                mask        = labels_flat != -100

                if mask.sum() == 0:
                    continue

                preds = torch.argmax(logits_flat[mask], dim=1).cpu().numpy()
                all_preds.extend(preds.tolist())
                all_trues.extend(labels_flat[mask].cpu().numpy().tolist())

        metrics    = compute_detailed_metrics(all_trues, all_preds)
        cls_report = classification_report(
            [id2label[x] for x in all_trues],
            [id2label[x] for x in all_preds],
            digits=4, zero_division=0,
        )
        return {
            "metrics":               metrics,
            "classification_report": cls_report,
            "all_preds":             all_preds,
            "all_trues":             all_trues,
        }

# ============================================================================
# HELPER: compute prototype embeddings with a temporary frozen model
# ============================================================================

def _compute_embeddings_batch(model, tokenizer, sentences, batch_size=8):
    """
    Mean-pool last hidden states.  Works with device_map='auto'.
    """
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(sentences), batch_size):
            batch_sents = sentences[i : i + batch_size]
            enc = tokenizer(
                batch_sents, padding=True, truncation=True,
                max_length=128, return_tensors="pt",
            )
            # Send to first device; device_map routes the rest
            first_dev = next(model.parameters()).device
            enc = {k: v.to(first_dev) for k, v in enc.items()}

            out         = model.model(**enc, output_hidden_states=True)
            last_hidden = out.hidden_states[-1]   # [B, L, H]

            mask_aligned = enc["attention_mask"].to(last_hidden.device)
            mask_exp     = mask_aligned.unsqueeze(-1).expand(last_hidden.size()).float()
            sum_emb      = torch.sum(last_hidden * mask_exp, dim=1)
            sum_mask     = torch.clamp(mask_exp.sum(dim=1), min=1e-9)
            sent_emb     = (sum_emb / sum_mask).cpu().float().numpy()
            embeddings.append(sent_emb)

            if (i // batch_size + 1) % 50 == 0:
                print(f"  Processed {i + len(batch_sents)}/{len(sentences)} sentences")

    return np.vstack(embeddings)

# ============================================================================
# MAIN
# ============================================================================

def main():
    set_seed()

    print(f"Primary device : {PRIMARY_DEVICE}")
    print(f"GPU count      : {torch.cuda.device_count()}")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            mem = torch.cuda.get_device_properties(i).total_memory / 1024**3
            print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  ({mem:.1f} GB)")

    # ── Data ──────────────────────────────────────────────────────────────────
    print("\nLoading data...")
    train_docs = load_jsonl(TRAIN_PATH)
    dev_docs   = load_jsonl(DEV_PATH)
    test_docs  = load_jsonl(TEST_PATH)

    train_sents, train_labels, _ = extract_data(train_docs)
    dev_sents,   dev_labels,   _ = extract_data(dev_docs)
    test_sents,  test_labels,  _ = extract_data(test_docs)

    print(f"Train: {len(train_sents)} docs | "
          f"Dev: {len(dev_sents)} | Test: {len(test_sents)}")

    # ── Tokenizer ─────────────────────────────────────────────────────────────
    print(f"\nLoading tokenizer from {MISTRAL_MODEL_NAME}...")
    tokenizer = AutoTokenizer.from_pretrained(
        MISTRAL_MODEL_NAME,
        trust_remote_code=True,
        use_fast=False,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token    = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "right"

    # ── Prototype computation with temporary model ────────────────────────────
    print("\nComputing class prototypes with base Mistral-7B...")

    if USE_QLORA:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit              = True,
            bnb_4bit_use_double_quant = True,
            bnb_4bit_quant_type       = "nf4",
            bnb_4bit_compute_dtype    = QLORA_COMPUTE_DTYPE,
        )
        temp_model = AutoModelForCausalLM.from_pretrained(
            MISTRAL_MODEL_NAME,
            quantization_config = bnb_config,
            device_map          = "auto",
            trust_remote_code   = True,
        )
    else:
        temp_model = AutoModelForCausalLM.from_pretrained(
            MISTRAL_MODEL_NAME,
            device_map    = "auto",
            torch_dtype   = QLORA_COMPUTE_DTYPE,
            trust_remote_code = True,
        )

    temp_model.eval()
    flat_train_sents  = [s for doc in train_sents for s in doc]
    flat_train_labels = np.array([l for doc in train_labels for l in doc], dtype=np.int64)

    train_embs = _compute_embeddings_batch(temp_model, tokenizer, flat_train_sents)

    proto_mgr = ClassPrototypeManager()
    proto_mgr.fit(train_embs, flat_train_labels)

    del temp_model
    torch.cuda.empty_cache()
    gc.collect()

    # ── Datasets ──────────────────────────────────────────────────────────────
    train_dataset = MistralLegalDataset(train_sents, train_labels, tokenizer)
    dev_dataset   = MistralLegalDataset(dev_sents,   dev_labels,   tokenizer)
    test_dataset  = MistralLegalDataset(test_sents,  test_labels,  tokenizer)

    # ── Model ─────────────────────────────────────────────────────────────────
    print(f"\nInitializing MistralLegalClassifier ({MISTRAL_MODEL_NAME})...")
    model = MistralLegalClassifier(head_device=PRIMARY_DEVICE)

    #  FIX 8: count only the classification heads for "non-backbone" params
    backbone_ids   = {id(p) for p in model.mistral.parameters()}
    head_params    = sum(
        p.numel() for p in model.parameters() if id(p) not in backbone_ids
    )
    lora_trainable = sum(
        p.numel() for p in model.mistral.parameters() if p.requires_grad
    )
    total_params   = sum(p.numel() for p in model.parameters())
    trainable_tot  = lora_trainable + head_params

    print(f"\nParameter summary:")
    print(f"  Total params       : {total_params:,}")
    print(f"  LoRA trainable     : {lora_trainable:,}")
    print(f"  Head trainable     : {head_params:,}")
    print(f"  Total trainable    : {trainable_tot:,} "
          f"({100.0 * trainable_tot / total_params:.2f}%)")

    # ── Training ──────────────────────────────────────────────────────────────
    trainer   = MistralTrainer(model, tokenizer, proto_mgr)
    best_ckpt = trainer.train(train_dataset, dev_dataset)

    # ── Load best checkpoint ──────────────────────────────────────────────────
    if best_ckpt and os.path.exists(best_ckpt):
        print(f"\nLoading best checkpoint: {best_ckpt}")
        ckpt = torch.load(best_ckpt, map_location="cpu")
        model.load_state_dict(ckpt["model_state_dict"], strict=False)
        print(f"  Best epoch macro-F1: {ckpt['macro_f1']:.4f}")

    # ── Final test evaluation ─────────────────────────────────────────────────
    print("\nFINAL TEST EVALUATION")
    test_results = trainer.evaluate(test_dataset)

    print(f"\nTest Macro-F1      : {test_results['metrics']['macro_f1']:.4f}")
    print(f"Minority Macro-F1  : {test_results['metrics']['minority_macro_f1']:.4f}")
    print(f"Weighted F1        : {test_results['metrics']['weighted_f1']:.4f}")
    print("\nClassification Report:")
    print(test_results["classification_report"])

    print("\nPer-Class F1 Scores:")
    for cls, f1_val in sorted(
        test_results["metrics"]["per_class_f1"].items(),
        key=lambda x: x[1], reverse=True
    ):
        support = test_results["metrics"]["per_class_support"][cls]
        marker  = "RARE" if cls in MINORITY_CLASSES else "    "
        print(f"  [{marker}] {cls:20s}: {f1_val:.4f}  (n={support})")

    # ── Save outputs ──────────────────────────────────────────────────────────
    results_summary = {
        "model":      MISTRAL_MODEL_NAME,
        "method":     "QLoRA + Prototype Learning",
        "lora_rank":  LORA_R,
        "lora_alpha": LORA_ALPHA,
        **{k: v for k, v in test_results["metrics"].items()
           if not isinstance(v, dict)},
    }
    pd.DataFrame([results_summary]).to_csv(
        os.path.join(OUT_DIR, "final_results.csv"), index=False)

    pd.DataFrame({
        "true_label": [id2label[x] for x in test_results["all_trues"]],
        "pred_label": [id2label[x] for x in test_results["all_preds"]],
        "correct":    [t == p for t, p in zip(
                           test_results["all_trues"],
                           test_results["all_preds"])],
    }).to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

    print(f"\nResults saved to {OUT_DIR}/")
    print("Mistral-7B Legal Classification Complete!")


if __name__ == "__main__":
    main()